In [ ]:
from ucimlrepo import fetch_ucirepo

ionosphere = fetch_ucirepo(id=52)

X = ionosphere.data.features
y = ionosphere.data.targets

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget counts:")
print(y.value_counts())

print("\nFirst five rows:")
print(X.head())

In [ ]:
import numpy as np

# Convert target to a 1D binary array
y = y["Class"].map({"b": 0, "g": 1}).to_numpy()

print("Target shape:", y.shape)
print("Classes:", np.unique(y, return_counts=True))

print("\nMissing values in X:")
print(X.isna().sum().sum())

print("\nConstant features:")
constant_cols = X.columns[X.nunique() <= 1].tolist()
print(constant_cols)

In [ ]:
from sklearn.model_selection import train_test_split

X = X.drop(columns=["Attribute2"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("X shape:", X.shape)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain classes:")
print(np.unique(y_train, return_counts=True))

print("\nTest classes:")
print(np.unique(y_test, return_counts=True))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

print("\nMean of scaled training features:")
print(np.round(X_train_scaled.mean(axis=0)[:5], 4))

print("\nStd of scaled training features:")
print(np.round(X_train_scaled.std(axis=0)[:5], 4))

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

kernels = ["linear", "rbf", "poly", "sigmoid"]

for kernel in kernels:
    model = SVC(
        kernel=kernel,
        C=1.0,
        probability=True,
        random_state=42,
    )

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"{kernel:8s}: {accuracy:.4f}")

In [ ]:
from svdynamics import AdaptiveCompositeKernel, SVDClassifier

adaptive_kernel = AdaptiveCompositeKernel(
    kernels=["linear", "rbf", "poly"]
)

model_adaptive = SVDClassifier(
    kernel=adaptive_kernel,
    C=1.0,
)

model_adaptive.fit(X_train_scaled, y_train)

y_pred_adaptive = model_adaptive.predict(X_test_scaled)

accuracy_adaptive = accuracy_score(
    y_test,
    y_pred_adaptive,
)

print("Adaptive accuracy:")
print(accuracy_adaptive)

print("\nLearned weights:")
print(adaptive_kernel.weights_)

print("\nAlignments:")
print(adaptive_kernel.alignments_)

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42,
)

splits = list(cv.split(X, y))

print("Number of splits:", len(splits))

train_idx, test_idx = splits[0]

print("First train size:", len(train_idx))
print("First test size:", len(test_idx))

print("\nFirst train classes:")
print(np.unique(y[train_idx], return_counts=True))

print("\nFirst test classes:")
print(np.unique(y[test_idx], return_counts=True))

In [ ]:
import pandas as pd

results = []

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):

    X_train_cv = X.iloc[train_idx]
    X_test_cv = X.iloc[test_idx]

    y_train_cv = y[train_idx]
    y_test_cv = y[test_idx]

    # Scale independently within each fold
    scaler = StandardScaler()

    X_train_cv = scaler.fit_transform(X_train_cv)
    X_test_cv = scaler.transform(X_test_cv)

    for kernel in ["linear", "rbf", "poly", "sigmoid"]:

        model = SVC(
            kernel=kernel,
            C=1.0,
            random_state=42,
        )

        model.fit(X_train_cv, y_train_cv)

        y_pred_cv = model.predict(X_test_cv)

        results.append(
            {
                "split": split_num,
                "model": kernel,
                "accuracy": accuracy_score(
                    y_test_cv,
                    y_pred_cv,
                ),
            }
        )

results_df = pd.DataFrame(results)

summary = (
    results_df
    .groupby("model")["accuracy"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

print(summary)

In [ ]:
adaptive_results = []

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):

    X_train_cv = X.iloc[train_idx]
    X_test_cv = X.iloc[test_idx]

    y_train_cv = y[train_idx]
    y_test_cv = y[test_idx]

    scaler = StandardScaler()

    X_train_cv = scaler.fit_transform(X_train_cv)
    X_test_cv = scaler.transform(X_test_cv)

    adaptive_kernel = AdaptiveCompositeKernel(
        kernels=["linear", "rbf", "poly"]
    )

    model = SVDClassifier(
        kernel=adaptive_kernel,
        C=1.0,
    )

    model.fit(X_train_cv, y_train_cv)

    y_pred_cv = model.predict(X_test_cv)

    adaptive_results.append(
        {
            "split": split_num,
            "accuracy": accuracy_score(
                y_test_cv,
                y_pred_cv,
            ),
            "linear_weight": adaptive_kernel.weights_[0],
            "rbf_weight": adaptive_kernel.weights_[1],
            "poly_weight": adaptive_kernel.weights_[2],
            "linear_alignment": adaptive_kernel.alignments_[0],
            "rbf_alignment": adaptive_kernel.alignments_[1],
            "poly_alignment": adaptive_kernel.alignments_[2],
        }
    )

adaptive_df = pd.DataFrame(adaptive_results)

print("Adaptive Composite accuracy:")
print(
    adaptive_df["accuracy"].agg(
        ["mean", "std", "min", "max"]
    )
)

print("\nMean learned weights:")
print(
    adaptive_df[
        [
            "linear_weight",
            "rbf_weight",
            "poly_weight",
        ]
    ].mean()
)

print("\nMean alignments:")
print(
    adaptive_df[
        [
            "linear_alignment",
            "rbf_alignment",
            "poly_alignment",
        ]
    ].mean()
)

In [ ]:
rbf_df = (
    results_df[results_df["model"] == "rbf"]
    [["split", "accuracy"]]
    .rename(columns={"accuracy": "rbf_accuracy"})
)

comparison_df = adaptive_df.merge(
    rbf_df,
    on="split",
)

comparison_df["difference"] = (
    comparison_df["accuracy"]
    - comparison_df["rbf_accuracy"]
)

print("Adaptive wins:")
print((comparison_df["difference"] > 0).sum())

print("\nTies:")
print((comparison_df["difference"] == 0).sum())

print("\nRBF wins:")
print((comparison_df["difference"] < 0).sum())

print("\nMean difference:")
print(comparison_df["difference"].mean())

print("\nMedian difference:")
print(comparison_df["difference"].median())

In [ ]:
from ucimlrepo import fetch_ucirepo

dry_bean = fetch_ucirepo(id=602)

X_bean = dry_bean.data.features
y_bean = dry_bean.data.targets

print("X shape:", X_bean.shape)
print("y shape:", y_bean.shape)

print("\nTarget counts:")
print(y_bean.value_counts())

print("\nMissing values:")
print(X_bean.isna().sum().sum())

print("\nConstant features:")
print(
    X_bean.columns[
        X_bean.nunique() <= 1
    ].tolist()
)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode target
encoder = LabelEncoder()
y_bean_encoded = encoder.fit_transform(
    y_bean["Class"]
)

print("Classes:")
print(encoder.classes_)

print("\nEncoded classes:")
print(
    np.unique(
        y_bean_encoded,
        return_counts=True,
    )
)

In [ ]:
from sklearn.model_selection import train_test_split

# Create a stratified 1,000-row sample
X_sample, _, y_sample, _ = train_test_split(
    X_bean,
    y_bean_encoded,
    train_size=1000,
    stratify=y_bean_encoded,
    random_state=42,
)

# Split the sample
X_train_bean, X_test_bean, y_train_bean, y_test_bean = (
    train_test_split(
        X_sample,
        y_sample,
        test_size=0.20,
        stratify=y_sample,
        random_state=42,
    )
)

# Scale
scaler = StandardScaler()

X_train_bean_scaled = scaler.fit_transform(
    X_train_bean
)

X_test_bean_scaled = scaler.transform(
    X_test_bean
)

# Adaptive kernel
adaptive_kernel = AdaptiveCompositeKernel(
    kernels=["linear", "rbf", "poly"]
)

model = SVDClassifier(
    kernel=adaptive_kernel,
    C=1.0,
)

model.fit(
    X_train_bean_scaled,
    y_train_bean,
)

y_pred = model.predict(
    X_test_bean_scaled
)

print("Accuracy:")
print(
    accuracy_score(
        y_test_bean,
        y_pred,
    )
)

print("\nLearned weights:")
print(adaptive_kernel.weights_)

print("\nAlignments:")
print(adaptive_kernel.alignments_)

In [ ]:
X_train_bean, X_test_bean, y_train_bean, y_test_bean = (
    train_test_split(
        X_bean,
        y_bean_encoded,
        test_size=0.20,
        stratify=y_bean_encoded,
        random_state=42,
    )
)

scaler = StandardScaler()

X_train_bean_scaled = scaler.fit_transform(
    X_train_bean
)

X_test_bean_scaled = scaler.transform(
    X_test_bean
)

for kernel in ["linear", "rbf", "poly", "sigmoid"]:

    model = SVC(
        kernel=kernel,
        C=1.0,
        random_state=42,
    )

    model.fit(
        X_train_bean_scaled,
        y_train_bean,
    )

    y_pred = model.predict(
        X_test_bean_scaled
    )

    print(
        f"{kernel:8s}: "
        f"{accuracy_score(y_test_bean, y_pred):.4f}"
    )

In [ ]:
adaptive_kernel_bean = AdaptiveCompositeKernel(
    kernels=["linear", "rbf", "poly"]
)

model_adaptive_bean = SVDClassifier(
    kernel=adaptive_kernel_bean,
    C=1.0,
)

model_adaptive_bean.fit(
    X_train_bean_scaled,
    y_train_bean,
)

y_pred_adaptive_bean = model_adaptive_bean.predict(
    X_test_bean_scaled
)

print("Adaptive accuracy:")
print(
    accuracy_score(
        y_test_bean,
        y_pred_adaptive_bean,
    )
)

print("\nLearned weights:")
print(adaptive_kernel_bean.weights_)

print("\nAlignments:")
print(adaptive_kernel_bean.alignments_)

In [ ]:
from ucimlrepo import fetch_ucirepo

spambase = fetch_ucirepo(id=94)

X_spam = spambase.data.features
y_spam = spambase.data.targets

print("X shape:", X_spam.shape)
print("y shape:", y_spam.shape)

print("\nTarget counts:")
print(y_spam.value_counts())

print("\nMissing values:")
print(X_spam.isna().sum().sum())

print("\nConstant features:")
print(
    X_spam.columns[
        X_spam.nunique() <= 1
    ].tolist()
)

In [ ]:
y_spam_encoded = y_spam["Class"].to_numpy()

X_train_spam, X_test_spam, y_train_spam, y_test_spam = (
    train_test_split(
        X_spam,
        y_spam_encoded,
        test_size=0.20,
        stratify=y_spam_encoded,
        random_state=42,
    )
)

scaler = StandardScaler()

X_train_spam_scaled = scaler.fit_transform(
    X_train_spam
)

X_test_spam_scaled = scaler.transform(
    X_test_spam
)

print("Train shape:", X_train_spam_scaled.shape)
print("Test shape:", X_test_spam_scaled.shape)

print("\nTrain classes:")
print(np.unique(y_train_spam, return_counts=True))

print("\nTest classes:")
print(np.unique(y_test_spam, return_counts=True))

In [ ]:
for kernel in ["linear", "rbf", "poly", "sigmoid"]:

    model = SVC(
        kernel=kernel,
        C=1.0,
        random_state=42,
    )

    model.fit(
        X_train_spam_scaled,
        y_train_spam,
    )

    y_pred = model.predict(
        X_test_spam_scaled
    )

    print(
        f"{kernel:8s}: "
        f"{accuracy_score(y_test_spam, y_pred):.4f}"
    )

In [ ]:
adaptive_kernel_spam = AdaptiveCompositeKernel(
    kernels=["linear", "rbf", "poly"]
)

model_adaptive_spam = SVDClassifier(
    kernel=adaptive_kernel_spam,
    C=1.0,
)

model_adaptive_spam.fit(
    X_train_spam_scaled,
    y_train_spam,
)

y_pred_adaptive_spam = model_adaptive_spam.predict(
    X_test_spam_scaled
)

print("Adaptive accuracy:")
print(
    accuracy_score(
        y_test_spam,
        y_pred_adaptive_spam,
    )
)

print("\nLearned weights:")
print(adaptive_kernel_spam.weights_)

print("\nAlignments:")
print(adaptive_kernel_spam.alignments_)

In [ ]:
from ucimlrepo import fetch_ucirepo

sonar = fetch_ucirepo(id=151)

X_sonar = sonar.data.features
y_sonar = sonar.data.targets

print("X shape:", X_sonar.shape)
print("y shape:", y_sonar.shape)

print("\nTarget counts:")
print(y_sonar.value_counts())

print("\nMissing values:")
print(X_sonar.isna().sum().sum())

print("\nConstant features:")
print(
    X_sonar.columns[
        X_sonar.nunique() <= 1
    ].tolist()
)

In [ ]:
y_sonar_encoded = (
    y_sonar["class"]
    .map({"R": 0, "M": 1})
    .to_numpy()
)

X_train_sonar, X_test_sonar, y_train_sonar, y_test_sonar = (
    train_test_split(
        X_sonar,
        y_sonar_encoded,
        test_size=0.20,
        stratify=y_sonar_encoded,
        random_state=42,
    )
)

scaler = StandardScaler()

X_train_sonar_scaled = scaler.fit_transform(
    X_train_sonar
)

X_test_sonar_scaled = scaler.transform(
    X_test_sonar
)

print("Train shape:", X_train_sonar_scaled.shape)
print("Test shape:", X_test_sonar_scaled.shape)

print("\nTrain classes:")
print(np.unique(y_train_sonar, return_counts=True))

print("\nTest classes:")
print(np.unique(y_test_sonar, return_counts=True))

In [ ]:
for kernel in ["linear", "rbf", "poly", "sigmoid"]:

    model = SVC(
        kernel=kernel,
        C=1.0,
        random_state=42,
    )

    model.fit(
        X_train_sonar_scaled,
        y_train_sonar,
    )

    y_pred = model.predict(
        X_test_sonar_scaled
    )

    print(
        f"{kernel:8s}: "
        f"{accuracy_score(y_test_sonar, y_pred):.4f}"
    )

In [ ]:
adaptive_kernel_sonar = AdaptiveCompositeKernel(
    kernels=["linear", "rbf", "poly"]
)

model_adaptive_sonar = SVDClassifier(
    kernel=adaptive_kernel_sonar,
    C=1.0,
)

model_adaptive_sonar.fit(
    X_train_sonar_scaled,
    y_train_sonar,
)

y_pred_adaptive_sonar = model_adaptive_sonar.predict(
    X_test_sonar_scaled
)

print("Adaptive accuracy:")
print(
    accuracy_score(
        y_test_sonar,
        y_pred_adaptive_sonar,
    )
)

print("\nLearned weights:")
print(adaptive_kernel_sonar.weights_)

print("\nAlignments:")
print(adaptive_kernel_sonar.alignments_)

In [ ]:
cv_sonar = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42,
)

splits_sonar = list(
    cv_sonar.split(
        X_sonar,
        y_sonar_encoded,
    )
)

print("Number of splits:", len(splits_sonar))

train_idx, test_idx = splits_sonar[0]

print("First train size:", len(train_idx))
print("First test size:", len(test_idx))

print("\nFirst train classes:")
print(
    np.unique(
        y_sonar_encoded[train_idx],
        return_counts=True,
    )
)

print("\nFirst test classes:")
print(
    np.unique(
        y_sonar_encoded[test_idx],
        return_counts=True,
    )
)

In [ ]:
sonar_results = []

for split_num, (train_idx, test_idx) in enumerate(
    splits_sonar,
    start=1,
):

    X_train_cv = X_sonar.iloc[train_idx]
    X_test_cv = X_sonar.iloc[test_idx]

    y_train_cv = y_sonar_encoded[train_idx]
    y_test_cv = y_sonar_encoded[test_idx]

    scaler = StandardScaler()

    X_train_cv = scaler.fit_transform(X_train_cv)
    X_test_cv = scaler.transform(X_test_cv)

    for kernel in ["linear", "rbf", "poly", "sigmoid"]:

        model = SVC(
            kernel=kernel,
            C=1.0,
            random_state=42,
        )

        model.fit(
            X_train_cv,
            y_train_cv,
        )

        y_pred_cv = model.predict(X_test_cv)

        sonar_results.append(
            {
                "split": split_num,
                "model": kernel,
                "accuracy": accuracy_score(
                    y_test_cv,
                    y_pred_cv,
                ),
            }
        )

sonar_results_df = pd.DataFrame(
    sonar_results
)

sonar_summary = (
    sonar_results_df
    .groupby("model")["accuracy"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

print(sonar_summary)

In [ ]:
sonar_adaptive_results = []

for split_num, (train_idx, test_idx) in enumerate(
    splits_sonar,
    start=1,
):

    X_train_cv = X_sonar.iloc[train_idx]
    X_test_cv = X_sonar.iloc[test_idx]

    y_train_cv = y_sonar_encoded[train_idx]
    y_test_cv = y_sonar_encoded[test_idx]

    scaler = StandardScaler()

    X_train_cv = scaler.fit_transform(X_train_cv)
    X_test_cv = scaler.transform(X_test_cv)

    adaptive_kernel = AdaptiveCompositeKernel(
        kernels=["linear", "rbf", "poly"]
    )

    model = SVDClassifier(
        kernel=adaptive_kernel,
        C=1.0,
    )

    model.fit(
        X_train_cv,
        y_train_cv,
    )

    y_pred_cv = model.predict(X_test_cv)

    sonar_adaptive_results.append(
        {
            "split": split_num,
            "accuracy": accuracy_score(
                y_test_cv,
                y_pred_cv,
            ),
            "linear_weight": adaptive_kernel.weights_[0],
            "rbf_weight": adaptive_kernel.weights_[1],
            "poly_weight": adaptive_kernel.weights_[2],
            "linear_alignment": adaptive_kernel.alignments_[0],
            "rbf_alignment": adaptive_kernel.alignments_[1],
            "poly_alignment": adaptive_kernel.alignments_[2],
        }
    )

sonar_adaptive_df = pd.DataFrame(
    sonar_adaptive_results
)

print("Adaptive Composite accuracy:")
print(
    sonar_adaptive_df["accuracy"].agg(
        ["mean", "std", "min", "max"]
    )
)

print("\nMean learned weights:")
print(
    sonar_adaptive_df[
        [
            "linear_weight",
            "rbf_weight",
            "poly_weight",
        ]
    ].mean()
)

print("\nMean alignments:")
print(
    sonar_adaptive_df[
        [
            "linear_alignment",
            "rbf_alignment",
            "poly_alignment",
        ]
    ].mean()
)